# Stage B — Neural Galactic Potential on Real IllustrisTNG Data

**Prerequisites:**
- Stage A complete ✓
- IllustrisTNG API key (free at tng-project.org) — OR run without key for mock data
- T4 GPU enabled (Runtime → Change runtime type → T4)

**What this notebook does:**
1. Downloads 5 real dark matter halos from IllustrisTNG
2. Builds density + potential fields from particle data
3. Trains SIREN with two-phase adaptive PINN (fixes Stage A instability)
4. Evaluates compression ratio and reconstruction error across all 5 halos
5. Generates the multi-halo benchmark table

In [1]:
# Cell 1 — Install dependencies
!pip install torch numpy scipy matplotlib h5py requests --quiet

# Optional but recommended for faster tree-based potential:
!pip install scikit-learn --quiet

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA:    {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:     {torch.cuda.get_device_name(0)}')


[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


PyTorch: 2.5.1+cu121
CUDA:    True
GPU:     NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [7]:
# Cell 2 — Upload project and set API key
import os, sys

# Upload the siren_grav_b folder via the file browser (left sidebar)
PROJECT_PATH = 'C:/Users/soham/Downloads/siren_grav_stage_a/siren_grav_b'
sys.path.insert(0, PROJECT_PATH)
os.chdir(PROJECT_PATH)

# Set your TNG API key here
# Get one free at: https://www.tng-project.org/users/register/
# Leave empty to run with mock data (for testing the pipeline)
os.environ['TNG_API_KEY'] = 'cac67c033e9150dc16273b7a0dda5699'   # <-- paste your key here

if os.environ['TNG_API_KEY']:
    print('API key set — will download real IllustrisTNG data')
else:
    print('No API key — will use mock NFW-like halos for testing')

API key set — will download real IllustrisTNG data


In [8]:
# Cell 3 — Quick unit conversion test
from src.units import TNGUnits, SimUnits
units = TNGUnits(redshift=0.0, h=0.6774)
print('Unit conversion: OK')

TNG Units initialised for z=0.0 (a=1.0000):
  pos  × 1.476233  → physical kpc
  mass × 1.4762e+10 → M_sun
  vel  × 1.000000  → km/s
Unit conversion: OK


In [11]:
# Cell 4 — Run full Stage B pipeline
# This cell runs everything: download → build → train → evaluate → plot
# Estimated time: 30-60 min on T4 GPU (mock data), 2-4 hrs (real TNG data)
!python experiments/run_stage_b.py

STAGE B — Neural Galactic Potential on Real IllustrisTNG Data
Device : cuda

STEP 1 — IllustrisTNG Data Download
TNG API connected. Available simulations: ['Illustris-1', 'Illustris-1-Dark', 'Illustris-2', 'Illustris-2-Dark', 'Illustris-3']

Fetching top 5 halos from TNG100-1 Snapshot 99...
DEBUG URL: https://www.tng-project.org/api/TNG100-1/snapshots/99/halos/
  API returned non-JSON response:
  Status: 200
  Content preview: 


<!DOCTYPE html>
<html>
  <head>
    <title>IllustrisTNG - Main</title>
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <meta name="description" content="The IllustrisTNG project. The next generation of cosmological hydrodynamical simulations of galaxy formation and evolution.">
    <meta name="author" content="Dylan Nelson for the TNG Collaboration.">
    <link href="/static/min/base_tng.css" rel="stylesheet" type="text/css" />
    
    <!-- HTML5 Shim and Respo
  Retry 1/3: Expecting value: line 4 column 1 (char 3)
  API returne

Traceback (most recent call last):
  File "C:\Users\soham\AppData\Roaming\Python\Python310\site-packages\requests\models.py", line 976, in json
    return complexjson.loads(self.text, **kwargs)
  File "c:\Program Files\Python310\lib\json\__init__.py", line 346, in loads
    return _default_decoder.decode(s)
  File "c:\Program Files\Python310\lib\json\decoder.py", line 337, in decode
    obj, end = self.raw_decode(s, idx=_w(s, 0).end())
  File "c:\Program Files\Python310\lib\json\decoder.py", line 355, in raw_decode
    raise JSONDecodeError("Expecting value", s, err.value) from None
json.decoder.JSONDecodeError: Expecting value: line 4 column 1 (char 3)

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "C:\Users\soham\Downloads\siren_grav_stage_a\siren_grav_b\experiments\run_stage_b.py", line 513, in <module>
    main()
  File "C:\Users\soham\Downloads\siren_grav_stage_a\siren_grav_b\experiments\run_stage_b.py", line 462, in

In [ ]:
# Cell 5 — Display plots
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob

OUTPUT_DIR = 'outputs/stage_b'

# Multi-halo benchmark
bench_path = f'{OUTPUT_DIR}/multi_halo_benchmark.png'
if os.path.exists(bench_path):
    img = mpimg.imread(bench_path)
    plt.figure(figsize=(14, 5))
    plt.imshow(img); plt.axis('off')
    plt.title('Stage B Multi-Halo Benchmark')
    plt.show()

# Per-halo density slices
density_plots = sorted(glob.glob(f'{OUTPUT_DIR}/halo_*/halo_*_density.png'))
if density_plots:
    fig, axes = plt.subplots(1, len(density_plots), figsize=(6*len(density_plots), 5))
    if len(density_plots) == 1:
        axes = [axes]
    for ax, path in zip(axes, density_plots):
        img = mpimg.imread(path)
        ax.imshow(img); ax.axis('off')
        ax.set_title(os.path.basename(path).replace('_density.png', ''))
    plt.tight_layout()
    plt.show()

In [ ]:
# Cell 6 — Read and print benchmark table
import json
import numpy as np

with open(f'{OUTPUT_DIR}/stage_b_results.json') as f:
    results = json.load(f)

print(f'{"Halo":>8} {"N_DM":>10} {"Raw MB":>8} {"SIREN MB":>9} {"Compress":>9} {"Rho err%":>9}')
print('-' * 60)
for halo_id, m in results.items():
    print(f'{halo_id:>8} {m["n_particles"]:>10,} {m["raw_size_mb"]:>8.1f} '
          f'{m["model_size_mb"]:>9.2f} {m["compression_ratio"]:>8.0f}x '
          f'{m["rho_rel_err_pct"]:>8.2f}%')

mean_comp = np.mean([m['compression_ratio'] for m in results.values()])
mean_err  = np.mean([m['rho_rel_err_pct']   for m in results.values()])
print(f'\nMean compression : {mean_comp:.0f}x')
print(f'Mean density err : {mean_err:.2f}%')

# Stage B verdict
print('\nSTAGE B VERDICT')
print('='*40)
ok_compress = mean_comp > 50
ok_err      = mean_err  < 10
print(f'Compression > 50x : {"PASS" if ok_compress else "FAIL"}  ({mean_comp:.0f}x)')
print(f'Density err  < 10% : {"PASS" if ok_err else "FAIL"}  ({mean_err:.2f}%)')
if ok_compress and ok_err:
    print('\nStage B COMPLETE — ready for Stage C (XAI + dark matter inference)')